# Experiment 19 — P3 KD: α=0.9, T=8.0## Setup| Item | Value ||------|-------|| **Pair** | P3 — `yolov8x-cls` (teacher) → `yolov8m-cls` (student) || **Dataset** | CIFAR10 (50 000 train / 10 000 val, 32×32, no resize) || **Pruning** | 50% unstructured magnitude (L1-norm, local scope) || **KD α** | 0.9 (soft-loss weight) || **KD T** | 8.0 (temperature) || **Epochs** | 5 || **Optimizer** | Adam, lr=1e-4 || **Batch size** | 128 || **Seed** | 42 |### PurposeTest logits distillation with near-pure soft-target regime (α=0.9) — minimal hard-label influence and t=8.0 spreads probability mass more evenly across classes..  T=8.0 spreads probability mass more evenly across classes.### Loss function$$\mathcal{L} = 0.9 \cdot T^2 \cdot \text{KL}\!\left(\sigma\!\left(\frac{z_s}{T}\right) \,\|\, \sigma\!\left(\frac{z_t}{T}\right)\right) + 0.1 \cdot \text{CE}(z_s, y)$$### Conditions evaluated1. **Teacher** — CIFAR10 fine-tuned `yolov8x-cls`, untouched  2. **Pruned (no train)** — Student immediately after 50% pruning  3. **Distilled (KD)** — Pruned student trained 5 epochs with KD loss  CE-only baseline is recorded in `exp_00_P3_baseline_CE.ipynb`.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────EXPERIMENT_ID = "exp_19"PAIR = "P3"TEACHER_CHECKPOINT = "data/cifar10_yolov8x_cls/runs/yolov8x_cls_cifar10_finetune/weights/best.pt"STUDENT_CHECKPOINT = "data/cifar10_yolov8m_cls/runs/yolov8m_cls_cifar10_finetune/weights/best.pt"TEACHER_CFG = "yolov8x-cls.yaml"STUDENT_CFG = "yolov8m-cls.yaml"DEVICE = "cuda"DATA_ROOT = "./data"IMAGE_SIZE = 32BATCH_SIZE = 128PRUNE_SPARSITY = 0.50KD_ALPHA = 0.9KD_TEMPERATURE = 8.0EPOCHS = 5LR = 1e-4SEED = 42RESULTS_CSV = "YOLO_logitKD_experiments/results.csv"KD_SAVE_PATH = "data/best_student_exp_19.pt"

## Imports and setup

In [ ]:
import copyimport csvimport randomimport sysimport timefrom pathlib import Pathimport torchfrom torch.utils.data import DataLoaderfrom torchvision import datasets, transformsfrom ultralytics import YOLOdef find_repo_root(start: Path) -> Path:    for candidate in [start, *start.parents]:        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():            return candidate    raise RuntimeError("Could not locate repository root containing src/")repo_root = find_repo_root(Path.cwd().resolve())src_path = repo_root / "src"if str(src_path) not in sys.path:    sys.path.insert(0, str(src_path))from chop import MaseGraphimport chop.passes as passesfrom chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolofrom mase_kd.core.losses import DistillationLossConfig, hard_label_ce_lossfrom mase_kd.vision.yolo_kd import YOLOLogitsDistillerif DEVICE == "cuda" and not torch.cuda.is_available():    DEVICE = "cpu"torch.manual_seed(SEED)random.seed(SEED)print(f"Repo root: {repo_root}")print(f"Device:    {DEVICE}")

## CIFAR10 dataloaders

In [ ]:
cifar_transform = transforms.Compose([transforms.ToTensor()])train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,  transform=cifar_transform, download=True)val_dataset   = datasets.CIFAR10(root=DATA_ROOT, train=False, transform=cifar_transform, download=True)train_loader = DataLoader(    train_dataset, batch_size=BATCH_SIZE, shuffle=True,    num_workers=2, pin_memory=True, drop_last=True,)val_loader = DataLoader(    val_dataset, batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True, drop_last=True,)print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

## Load CIFAR10-fine-tuned teacher (yolov8x-cls)

In [ ]:
ultra_teacher = YOLO(TEACHER_CHECKPOINT)nc = ultra_teacher.model.yaml.get("nc", 10)teacher_cls_model = MaseYoloClassificationModel(cfg=TEACHER_CFG, nc=nc)teacher_cls_model = patch_yolo(teacher_cls_model)teacher_cls_model.load_state_dict(ultra_teacher.model.state_dict())teacher_cls_model = teacher_cls_model.to(DEVICE)teacher_cls_model.eval()print(f"Teacher loaded: {TEACHER_CHECKPOINT}  (nc={nc})")

## Build student from checkpoint and prune (50% sparsity)

In [ ]:
ultra_student = YOLO(STUDENT_CHECKPOINT)student_seed = MaseYoloClassificationModel(cfg=STUDENT_CFG, nc=nc)student_seed = patch_yolo(student_seed)student_seed.load_state_dict(ultra_student.model.state_dict())mg = MaseGraph(student_seed)mg, _ = passes.init_metadata_analysis_pass(mg)trace_input = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)placeholder_names = [n.name for n in mg.fx_graph.nodes if n.op == "placeholder"]dummy_in = {name: trace_input for name in placeholder_names}mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_in})pruning_config = {    "weight":     {"sparsity": PRUNE_SPARSITY, "method": "l1-norm", "scope": "local"},    "activation": {"sparsity": PRUNE_SPARSITY, "method": "l1-norm", "scope": "local"},}mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)student_cls_model = mg.model.to(DEVICE)pruned_snapshot = copy.deepcopy(student_cls_model).to(DEVICE)pruned_snapshot.eval()print(f"Student loaded: {STUDENT_CHECKPOINT}")print(f"Pruning complete ({PRUNE_SPARSITY*100:.0f}% sparsity)")

## Evaluate pruned model (no training)

In [ ]:
@torch.no_grad()def evaluate_model(model, loader, device):    model.eval()    batches = samples = correct = 0    total_ce = total_ms = 0.0    for images, labels in loader:        images, labels = images.to(device), labels.to(device)        if device == "cuda":            torch.cuda.synchronize()        t0 = time.perf_counter()        outputs = model(images)        if device == "cuda":            torch.cuda.synchronize()        t1 = time.perf_counter()        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])        if logits is None or logits.numel() == 0:            continue        total_ms += (t1 - t0) * 1000.0        batches += 1        samples += images.shape[0]        if logits.shape[1] > int(labels.max().item()):            correct += int((logits.argmax(dim=1) == labels).sum().item())            total_ce += hard_label_ce_loss(logits, labels).item()    return {        "top1_acc": correct / max(samples, 1),        "avg_ce_loss": total_ce / max(batches, 1),        "avg_forward_ms": total_ms / max(batches, 1),        "samples": samples,        "batches": batches,    }pruned_metrics = evaluate_model(pruned_snapshot, val_loader, DEVICE)print(f"Pruned (no train) — top1: {pruned_metrics['top1_acc']*100:.2f}%  CE: {pruned_metrics['avg_ce_loss']:.4f}")

## Knowledge Distillation (α=0.9, T=8.0)

In [ ]:
kd_config = DistillationLossConfig(alpha=KD_ALPHA, temperature=KD_TEMPERATURE)optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=LR)distiller = YOLOLogitsDistiller(    teacher=teacher_cls_model,    student=student_cls_model,    kd_config=kd_config,    device=DEVICE,    train_loader=train_loader,    val_loader=val_loader,    optimizer=optimizer,    num_train_epochs=EPOCHS,    eval_teacher=True,)train_history = distiller.train(save_path=KD_SAVE_PATH)loss_history = train_history["total_loss"]top1_history = train_history["top1_acc"]top5_history = train_history["top5_acc"]# Restore best student weights before evaluationstudent_cls_model.load_state_dict(torch.load(KD_SAVE_PATH), strict=False)print(f"Best student weights restored from {KD_SAVE_PATH}")

## Evaluation: teacher vs pruned vs distilled (CIFAR10 val)

In [ ]:
eval_results = distiller.evaluate()teacher_metrics = eval_results.get("teacher")student_metrics = eval_results["student"]val_kd_loss = eval_results["val_kd_loss"]kd_batches = eval_results["kd_batches"]print(f"{'Model':<40} {'Top-1':>8} {'CE Loss':>10}")print(f"{'─'*40} {'─'*8} {'─'*10}")if teacher_metrics:    print(f"{'Teacher (yolov8x-cls)':<40} {teacher_metrics['top1_acc']*100:>7.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")print(f"{'Pruned (no train, 50%)':<40} {pruned_metrics['top1_acc']*100:>7.2f}% {pruned_metrics['avg_ce_loss']:>10.4f}")print(f"{f'Distilled (α=0.9, T=8.0)':<40} {student_metrics['top1_acc']*100:>7.2f}% {student_metrics['avg_ce_loss']:>10.4f}")print(f"\nVal KD loss: {val_kd_loss:.6f} ({kd_batches} batches)")

## Save results to CSV

In [ ]:
row = {    "pair": PAIR,    "alpha": KD_ALPHA,    "temperature": KD_TEMPERATURE,    "lr": LR,    "epochs": EPOCHS,    "sparsity": PRUNE_SPARSITY,    "batch_size": BATCH_SIZE,    "seed": SEED,    "teacher_top1": round(teacher_metrics["top1_acc"] * 100, 4) if teacher_metrics else "",    "pruned_top1": round(pruned_metrics["top1_acc"] * 100, 4),    "ce_only_top1": "",    "kd_top1": round(student_metrics["top1_acc"] * 100, 4),    "kd_gain_vs_ce": "",    "teacher_ce_loss": round(teacher_metrics["avg_ce_loss"], 6) if teacher_metrics else "",    "pruned_ce_loss": round(pruned_metrics["avg_ce_loss"], 6),    "ce_only_ce_loss": "",    "kd_ce_loss": round(student_metrics["avg_ce_loss"], 6),    "val_kd_loss": round(val_kd_loss, 6),    "notebook": EXPERIMENT_ID,}csv_path = Path(RESULTS_CSV)file_exists = csv_path.exists() and csv_path.stat().st_size > 0with open(csv_path, "a", newline="") as f:    writer = csv.DictWriter(f, fieldnames=row.keys())    if not file_exists:        writer.writeheader()    writer.writerow(row)print(f"Results appended to {RESULTS_CSV}")

## Training curves

In [ ]:
import matplotlib.pyplot as pltimport mathfig, axes = plt.subplots(1, 2, figsize=(14, 5))fig.suptitle(f"{EXPERIMENT_ID} — KD α={KD_ALPHA}, T={KD_TEMPERATURE} (P3, 50% pruned)", fontsize=13)batches_per_epoch = len(loss_history) // EPOCHSax = axes[0]ax.plot(loss_history, alpha=0.8, linewidth=0.8)for e in range(1, EPOCHS):    ax.axvline(e * batches_per_epoch, color="gray", ls="--", lw=0.6, alpha=0.5)ax.set_xlabel("Batch")ax.set_ylabel("KD Loss")ax.set_title("Training Loss")ax.grid(True, alpha=0.3)ax = axes[1]clean_top1 = [v if not math.isnan(v) else float("nan") for v in top1_history]ax.plot(clean_top1, alpha=0.8, linewidth=0.8)for e in range(1, EPOCHS):    ax.axvline(e * batches_per_epoch, color="gray", ls="--", lw=0.6, alpha=0.5)ax.set_xlabel("Batch")ax.set_ylabel("Top-1 Accuracy")ax.set_title("Training Top-1")ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%"))ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()